# 9 WorkFlow Analista Senior

### 9.5 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
enriquecer
<br>El Analista Sr corre sus scripts en máquinas virtuales de al menos 128 GB de RAM en Toronto, creando una virtual machine para cada corrida.
<br>Estas virtual machines se auto-suicidarán a los 30 minutos de haber terminado de procesar.

## 9.7  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Wed Nov 11 21:36:50 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671294,35.9,1479638,79.1,1479638,79.1
Vcells,1242867,9.5,8388608,64.0,1978693,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros


In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 415517 # Mi 2da semilla

PARAM$experimento <- 9103
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.7.1   Preprocesamiento del dataset

#### 9.7.1.1  DT incorporar dataset

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.7.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis interpolando (Problema #01) (modificación)
setorder( dataset, numero_de_cliente, foto_mes )

PARAM$dataset_metadata$entity_id <- "numero_de_cliente"
PARAM$CA$metodo <- "EstadisticaClasica"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.7.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.780952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
# funcion provista por alumnos amantes de la Estadistica Clasica

drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(get(campo), na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}



In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

#### 9.7.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [27]:
if( !require("lubridate")) install.packages("lubridate", repos = "http://cran.us.r-project.org")
require("lubridate")

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

In [29]:
# modificación de la función agregar variables intra-mes
AgregarVariables_IntraMes <- function(dataset) {
  cat( "inicio AgregarVariables_IntraMes()\n")
  gc(verbose= FALSE)
  # INICIO de la seccion donde se deben hacer cambios con variables nuevas

  # el mes 1,2, ..12
  if( atributos_presentes( c("foto_mes") ))
    dataset[, kmes := foto_mes %% 100]

  # creo un ctr_quarter que tenga en cuenta cuando
  # los clientes hace 3 menos meses que estan
  if( atributos_presentes( c("ctrx_quarter") ))
    dataset[, ctrx_quarter_normalizado := as.numeric(ctrx_quarter) ]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 1, ctrx_quarter_normalizado := ctrx_quarter * 5]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 2, ctrx_quarter_normalizado := ctrx_quarter * 2]

  if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
    dataset[cliente_antiguedad == 3, ctrx_quarter_normalizado := ctrx_quarter * 1.2]

  # variable extraida de una tesis de maestria de Irlanda
  if( atributos_presentes( c("mpayroll", "cliente_edad") ))
    dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

  # se crean los nuevos campos para MasterCard y Visa, teniendo en cuenta los NA's
  if( atributos_presentes( c("Master_status", "Visa_status") ))
  {
    dataset[, vm_status01 := pmax(Master_status, Visa_status, na.rm = TRUE)]
    dataset[, vm_status02 := Master_status + Visa_status]

    dataset[, vm_status03 := pmax(
      ifelse(is.na(Master_status), 10, Master_status),
      ifelse(is.na(Visa_status), 10, Visa_status)
    )]

    dataset[, vm_status04 := ifelse(is.na(Master_status), 10, Master_status)
      + ifelse(is.na(Visa_status), 10, Visa_status)]

    dataset[, vm_status05 := ifelse(is.na(Master_status), 10, Master_status)
      + 100 * ifelse(is.na(Visa_status), 10, Visa_status)]

    dataset[, vm_status06 := ifelse(is.na(Visa_status),
      ifelse(is.na(Master_status), 10, Master_status),
      Visa_status
    )]

    dataset[, mv_status07 := ifelse(is.na(Master_status),
      ifelse(is.na(Visa_status), 10, Visa_status),
      Master_status
    )]
  }

  # combino MasterCard y Visa (originales de la catedra presentes en Jr)
  if( atributos_presentes( c("Master_mfinanciacion_limite", "Visa_mfinanciacion_limite") ))
    dataset[, vm_mfinanciacion_limite := rowSums(cbind(Master_mfinanciacion_limite, Visa_mfinanciacion_limite), na.rm = TRUE)]

  if( atributos_presentes( c("Master_Fvencimiento", "Visa_Fvencimiento") ))
    dataset[, vm_Fvencimiento := pmin(Master_Fvencimiento, Visa_Fvencimiento, na.rm = TRUE)]

  if( atributos_presentes( c("Master_msaldototal", "Visa_msaldototal") ))
    dataset[, vm_msaldototal := rowSums(cbind(Master_msaldototal, Visa_msaldototal), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mlimitecompra", "Visa_mlimitecompra") ))
    dataset[, vm_mlimitecompra := rowSums(cbind(Master_mlimitecompra, Visa_mlimitecompra), na.rm = TRUE)]

  if( atributos_presentes( c("Master_fultimo_cierre", "Visa_fultimo_cierre") ))
    dataset[, vm_fultimo_cierre := pmax(Master_fultimo_cierre, Visa_fultimo_cierre, na.rm = TRUE)]

  if( atributos_presentes( c("Master_fechaalta", "Visa_fechaalta") ))
    dataset[, vm_fechaalta := pmax(Master_fechaalta, Visa_fechaalta, na.rm = TRUE)]

  if( atributos_presentes( c("Master_mconsumototal", "Visa_mconsumototal") ))
    dataset[, vm_mconsumototal := rowSums(cbind(Master_mconsumototal, Visa_mconsumototal), na.rm = TRUE)]

  if( atributos_presentes( c("Master_cconsumos", "Visa_cconsumos") ))
    dataset[, vm_cconsumos := rowSums(cbind(Master_cconsumos, Visa_cconsumos), na.rm = TRUE)]

  if( atributos_presentes( c("Master_mpagominimo", "Visa_mpagominimo") ))
    dataset[, vm_mpagominimo := rowSums(cbind(Master_mpagominimo, Visa_mpagominimo), na.rm = TRUE)]

  # ratios originales de la catedra presentes en Jr
  if( atributos_presentes( c("Master_mlimitecompra", "vm_mlimitecompra") ))
    dataset[, vmr_Master_mlimitecompra := Master_mlimitecompra / vm_mlimitecompra]

  if( atributos_presentes( c("Visa_mlimitecompra", "vm_mlimitecompra") ))
    dataset[, vmr_Visa_mlimitecompra := Visa_mlimitecompra / vm_mlimitecompra]

  if( atributos_presentes( c("vm_msaldototal", "vm_mlimitecompra") ))
    dataset[, vmr_msaldototal := vm_msaldototal / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mconsumototal", "vm_mlimitecompra") ))
    dataset[, vmr_mconsumototal := vm_mconsumototal / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagominimo", "vm_mlimitecompra") ))
    dataset[, vmr_mpagominimo := vm_mpagominimo / vm_mlimitecompra]

  # ============================================================================
  # NUEVAS VARIABLES INTRA-MES DISEÑADAS PARA LAS 55 COLUMNAS DE JUNIOR
  # ============================================================================

  # 1. Tarjetas (vision interna del banco: tenencia de plasticos, trxs y consumos)
  if( atributos_presentes( c("ctarjeta_visa", "ctarjeta_master") ))
    dataset[, vm_ctarjetas_activas := rowSums(cbind(ctarjeta_visa, ctarjeta_master), na.rm = TRUE)]

  if( atributos_presentes( c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones") ))
    dataset[, vm_ctrx_banco := rowSums(cbind(ctarjeta_visa_transacciones, ctarjeta_master_transacciones), na.rm = TRUE)]

  if( atributos_presentes( c("mtarjeta_visa_consumo", "mtarjeta_master_consumo") ))
    dataset[, vm_mconsumo_banco := rowSums(cbind(mtarjeta_visa_consumo, mtarjeta_master_consumo), na.rm = TRUE)]

  if( atributos_presentes( c("vm_mconsumo_banco", "vm_ctrx_banco") ))
    dataset[, vmr_ticket_medio_banco := vm_mconsumo_banco / (vm_ctrx_banco + 1)]

  if( atributos_presentes( c("vm_mconsumo_banco", "vm_mlimitecompra") ))
    dataset[, vmr_consumo_banco_limite := vm_mconsumo_banco / vm_mlimitecompra]

  if( atributos_presentes( c("vm_mpagominimo", "vm_msaldototal") ))
    dataset[, vmr_pagominimo_saldototal := vm_mpagominimo / vm_msaldototal]

  # 2. Liquidez, Endeudamiento y Flujo de Ingresos
  if( atributos_presentes( c("mcuenta_corriente", "mcaja_ahorro") )) {
    dataset[, m_liquidez_cuentas := rowSums(cbind(mcuenta_corriente, mcaja_ahorro), na.rm = TRUE)]
    dataset[, r_caja_ahorro_liquidez := mcaja_ahorro / (abs(m_liquidez_cuentas) + 1)]
  }

  if( atributos_presentes( c("mprestamos_personales", "vm_msaldototal") ))
    dataset[, m_deuda_total := rowSums(cbind(mprestamos_personales, vm_msaldototal), na.rm = TRUE)]

  if( atributos_presentes( c("mpayroll", "mtransferencias_recibidas") ))
    dataset[, m_flujo_ingresos := rowSums(cbind(mpayroll, mtransferencias_recibidas), na.rm = TRUE)]

  if( atributos_presentes( c("m_deuda_total", "mcuentas_saldo") ))
    dataset[, r_deuda_sobre_saldo := m_deuda_total / (abs(mcuentas_saldo) + 1)]

  if( atributos_presentes( c("m_deuda_total", "m_flujo_ingresos") ))
    dataset[, r_deuda_sobre_ingresos := m_deuda_total / (m_flujo_ingresos + 1)]

  if( atributos_presentes( c("vm_mconsumo_banco", "m_flujo_ingresos") ))
    dataset[, r_consumo_sobre_ingresos := vm_mconsumo_banco / (m_flujo_ingresos + 1)]

  # 3. Rentabilidad y Friccion por Comisiones / Mantenimiento
  if( atributos_presentes( c("mactivos_margen", "mpasivos_margen") ))
    dataset[, m_margen_total := rowSums(cbind(mactivos_margen, mpasivos_margen), na.rm = TRUE)]

  if( atributos_presentes( c("mrentabilidad", "mrentabilidad_annual") ))
    dataset[, r_rentabilidad_mes_vs_anual := mrentabilidad / (abs(mrentabilidad_annual) / 12 + 1)]

  if( atributos_presentes( c("mrentabilidad", "cproductos") ))
    dataset[, mrentabilidad_por_producto := mrentabilidad / (cproductos + 1)]

  if( atributos_presentes( c("mcomisiones", "cproductos") ))
    dataset[, mcomisiones_por_producto := mcomisiones / (cproductos + 1)]

  if( atributos_presentes( c("mcomisiones_mantenimiento", "mcuentas_saldo") ))
    dataset[, r_mantenimiento_sobre_saldo := mcomisiones_mantenimiento / (abs(mcuentas_saldo) + 1)]

  # 4. Actividad Transaccional e Inactividad (Separador de Cliente Zombi vs Baja)
  if( atributos_presentes( c("vm_ctrx_banco", "chomebanking_transacciones", "ccallcenter_transacciones", "cpayroll_trx") ))
    dataset[, c_operaciones_mes := rowSums(cbind(vm_ctrx_banco, chomebanking_transacciones, ccallcenter_transacciones, cpayroll_trx), na.rm = TRUE)]

  if( atributos_presentes( c("ctrx_quarter", "vm_ctrx_banco", "cpayroll_trx", "chomebanking_transacciones", "mttarjeta_visa_debitos_automaticos") )) {
    dataset[, score_inactividad := as.integer(ctrx_quarter == 0) +
                                   as.integer(vm_ctrx_banco == 0) +
                                   as.integer(cpayroll_trx == 0) +
                                   as.integer(chomebanking_transacciones == 0) +
                                   as.integer(mttarjeta_visa_debitos_automaticos == 0)]
  }

  if( atributos_presentes( c("cliente_antiguedad", "cliente_edad") ))
    dataset[, r_antiguedad_edad := cliente_antiguedad / ((cliente_edad - 18) * 12 + 1)]

 # ============================================================================
  # 5. FE 2da GENERACION (Basado en el Top 20 de impo.txt de WF9102)
  # ============================================================================

  # A. Uso de descubierto y precursores de cierre de tarjetas
  if( atributos_presentes( c("mcuentas_saldo", "cdescubierto_preacordado") )) {
    dataset[, flag_usa_descubierto := as.integer(mcuentas_saldo < 0)]
    dataset[, m_descubierto_utilizado := ifelse(mcuentas_saldo < 0, -mcuentas_saldo, 0)]
    dataset[, r_descubierto_sin_acuerdo := as.integer(mcuentas_saldo < 0 & cdescubierto_preacordado == 0)]
  }

  if( atributos_presentes( c("Visa_status", "Master_status") )) {
    dataset[, flag_tarjeta_en_cierre := as.integer(
      (!is.na(Visa_status) & Visa_status >= 6) | (!is.na(Master_status) & Master_status >= 6)
    )]
    dataset[, flag_ambas_tarjetas_na := as.integer(is.na(Visa_status) & is.na(Master_status))]
  }

  # Concentracion de productos y antiguedad de plasticos
  if( atributos_presentes( c("c_operaciones_mes", "cproductos") ))
    dataset[, r_operaciones_por_producto := c_operaciones_mes / (cproductos + 1)]

  if( atributos_presentes( c("Visa_fechaalta", "Master_fechaalta") ))
    dataset[, vm_fechaalta_min := pmin(Visa_fechaalta, Master_fechaalta, na.rm = TRUE)]

  # B. Percentiles Intra-Mes Cero-Fijo (Sin azar, Problema #02 Grupo A) para el Top Monetario
  cols_rank_top <- intersect(
    c("mcuentas_saldo", "mcaja_ahorro", "mcuenta_corriente", "m_liquidez_cuentas",
      "m_flujo_ingresos", "m_deuda_total", "vm_mconsumo_banco", "mtarjeta_visa_consumo",
      "mpayroll", "mpasivos_margen"),
    colnames(dataset)
  )

  for (campo in cols_rank_top) {
    dataset[, paste0(campo, "_rk0") := 0.0]
    dataset[get(campo) > 0, paste0(campo, "_rk0") :=
      frank(get(campo), ties.method = "average") / .N, by = foto_mes]
    dataset[get(campo) < 0, paste0(campo, "_rk0") :=
      -frank(-get(campo), ties.method = "average") / .N, by = foto_mes]
  }
  # ============================================================================
  # VALVULAS DE SEGURIDAD DE LA CATEDRA (Inf -> NA, NaN -> 0)
  # ============================================================================
  infinitos <- lapply(
    names(dataset),
    function(.name) dataset[, sum(is.infinite(get(.name)))]
  )

  infinitos_qty <- sum(unlist(infinitos))
  if (infinitos_qty > 0) {
    cat(
      "ATENCION, hay", infinitos_qty,
      "valores infinitos en tu dataset. Seran pasados a NA\n"
    )
    dataset[mapply(is.infinite, dataset)] <<- NA
  }

  nans <- lapply(
    names(dataset),
    function(.name) dataset[, sum(is.nan(get(.name)))]
  )

  nans_qty <- sum(unlist(nans))
  if (nans_qty > 0) {
    cat(
      "ATENCION, hay", nans_qty,
      "valores NaN 0/0 en tu dataset. Seran pasados arbitrariamente a 0\n"
    )
    cat("Si no te gusta la decision, modifica a gusto el programa!\n\n")
    dataset[mapply(is.nan, dataset)] <<- 0
  }

  cat( "fin AgregarVariables_IntraMes()\n")
}


In [30]:
# agrego las variables intra mes

AgregarVariables_IntraMes(dataset)

inicio AgregarVariables_IntraMes()
ATENCION, hay 18 valores infinitos en tu dataset. Seran pasados a NA
ATENCION, hay 265528 valores NaN 0/0 en tu dataset. Seran pasados arbitrariamente a 0
Si no te gusta la decision, modifica a gusto el programa!

fin AgregarVariables_IntraMes()


In [31]:
# visualizo las columas del dataset a esta etapa
ncol(dataset)
colnames(dataset)

[1] 117

[1] "numero_de_cliente"                  "foto_mes"                          
  [3] "internet"                           "cliente_edad"                      
  [5] "cliente_antiguedad"                 "mrentabilidad"                     
  [7] "mrentabilidad_annual"               "mcomisiones"                       
  [9] "mactivos_margen"                    "mpasivos_margen"                   
 [11] "cproductos"                         "mcuenta_corriente"                 
 [13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
 [15] "mcuentas_saldo"                     "ctarjeta_visa"                     
 [17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
 [19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
 [21] "mtarjeta_master_consumo"            "cprestamos_personales"             
 [23] "mprestamos_personales"              "cpayroll_trx"                      
 [25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
 [27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
 [29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
 [31] "ccallcenter_transacciones"          "thomebanking"                      
 [33] "chomebanking_transacciones"         "ctrx_quarter"                      
 [35] "Master_status"                      "Master_mfinanciacion_limite"       
 [37] "Master_Fvencimiento"                "Master_msaldototal"                
 [39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
 [41] "Master_fechaalta"                   "Master_mconsumototal"              
 [43] "Master_cconsumos"                   "Master_mpagominimo"                
 [45] "Visa_status"                        "Visa_mfinanciacion_limite"         
 [47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
 [49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
 [51] "Visa_fechaalta"                     "Visa_mconsumototal"                
 [53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
 [55] "clase_ternaria"                     "kmes"                              
 [57] "ctrx_quarter_normalizado"           "mpayroll_sobre_edad"               
 [59] "vm_status01"                        "vm_status02"                       
 [61] "vm_status03"                        "vm_status04"                       
 [63] "vm_status05"                        "vm_status06"                       
 [65] "mv_status07"                        "vm_mfinanciacion_limite"           
 [67] "vm_Fvencimiento"                    "vm_msaldototal"                    
 [69] "vm_mlimitecompra"                   "vm_fultimo_cierre"                 
 [71] "vm_fechaalta"                       "vm_mconsumototal"                  
 [73] "vm_cconsumos"                       "vm_mpagominimo"                    
 [75] "vmr_Master_mlimitecompra"           "vmr_Visa_mlimitecompra"            
 [77] "vmr_msaldototal"                    "vmr_mconsumototal"                 
 [79] "vmr_mpagominimo"                    "vm_ctarjetas_activas"              
 [81] "vm_ctrx_banco"                      "vm_mconsumo_banco"                 
 [83] "vmr_ticket_medio_banco"             "vmr_consumo_banco_limite"          
 [85] "vmr_pagominimo_saldototal"          "m_liquidez_cuentas"                
 [87] "r_caja_ahorro_liquidez"             "m_deuda_total"                     
 [89] "m_flujo_ingresos"                   "r_deuda_sobre_saldo"               
 [91] "r_deuda_sobre_ingresos"             "r_consumo_sobre_ingresos"          
 [93] "m_margen_total"                     "r_rentabilidad_mes_vs_anual"       
 [95] "mrentabilidad_por_producto"         "mcomisiones_por_producto"          
 [97] "r_mantenimiento_sobre_saldo"        "c_operaciones_mes"                 
 [99] "score_inactividad"                  "r_antiguedad_edad"                 
[1

#### 9.7.1.4  FEhist Feature Engineering historico

El Feature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [32]:
if( !require("Rcpp")) install.packages("Rcpp", repos = "http://cran.us.r-project.org")
require("Rcpp")

Loading required package: Rcpp



In [33]:
# se calculan para los 6 meses previos el minimo, maximo y
#  tendencia calculada con cuadrados minimos
# la formula de calculo de la tendencia puede verse en
#  https://stats.libretexts.org/Bookshelves/Introductory_Statistics/Book%3A_Introductory_Statistics_(Shafer_and_Zhang)/10%3A_Correlation_and_Regression/10.04%3A_The_Least_Squares_Regression_Line
# para la maxíma velocidad esta funcion esta escrita en lenguaje C,
# y no en la porqueria de R o Python

cppFunction("NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde )
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) )
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      {
        xsum  += x[h] ;
        ysum  += y[h] ;
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ;
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}")


In [34]:
# calcula la tendencia de las variables cols de los ultimos 6 meses
# la tendencia es la pendiente de la recta que ajusta por cuadrados minimos
# La funcionalidad de ratioavg es autoria de  Daiana Sparta,  UAustral  2021

TendenciaYmuchomas <- function(
    dataset, cols, ventana = 6, tendencia = TRUE,
    minimo = TRUE, maximo = TRUE, promedio = TRUE,
    ratioavg = FALSE, ratiomax = FALSE) {
  gc(verbose= FALSE)
  # Esta es la cantidad de meses que utilizo para la historia
  ventana_regresion <- ventana

  last <- nrow(dataset)

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids <- dataset[ , numero_de_cliente ]

  vector_desde <- seq(
    -ventana_regresion + 2,
    nrow(dataset) - ventana_regresion + 1
  )

  vector_desde[1:ventana_regresion] <- 1

  for (i in 2:last) {
    if (vector_ids[i - 1] != vector_ids[i]) {
      vector_desde[i] <- i
    }
  }
  for (i in 2:last) {
    if (vector_desde[i] < vector_desde[i - 1]) {
      vector_desde[i] <- vector_desde[i - 1]
    }
  }

  for (campo in cols) {
    nueva_col <- fhistC(dataset[, get(campo)], vector_desde)

    if (tendencia) {
      dataset[, paste0(campo, "_tend", ventana) :=
        nueva_col[(0 * last + 1):(1 * last)]]
    }

    if (minimo) {
      dataset[, paste0(campo, "_min", ventana) :=
        nueva_col[(1 * last + 1):(2 * last)]]
    }

    if (maximo) {
      dataset[, paste0(campo, "_max", ventana) :=
        nueva_col[(2 * last + 1):(3 * last)]]
    }

    if (promedio) {
      dataset[, paste0(campo, "_avg", ventana) :=
        nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratioavg) {
      dataset[, paste0(campo, "_ratioavg", ventana) :=
        get(campo) / nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratiomax) {
      dataset[, paste0(campo, "_ratiomax", ventana) :=
        get(campo) / nueva_col[(2 * last + 1):(3 * last)]]
    }
  }
}


In [35]:
# Feature Engineering Historico (modificación)
setorder(dataset, numero_de_cliente, foto_mes)

# CELDA 36: Lags 1 y 2 + Deltas 1 y 2 (Ganador en 202109 segun Problemas #04 y #05)
setorder(dataset, numero_de_cliente, foto_mes)

cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"), by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"), by = numero_de_cliente, .SDcols = cols_lagueables]

for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


In [36]:
# parametros de Feature Engineering Historico de Tendencias (modificación)
# CELDA 37: Solo la pendiente de 6 meses (sin min6 ni max6 que distorsionan post-aguinaldo)
PARAM$FE_hist$Tendencias$run <- TRUE
PARAM$FE_hist$Tendencias$ventana <- 6
PARAM$FE_hist$Tendencias$tendencia <- TRUE
PARAM$FE_hist$Tendencias$minimo <- FALSE
PARAM$FE_hist$Tendencias$maximo <- FALSE
PARAM$FE_hist$Tendencias$promedio <- FALSE
PARAM$FE_hist$Tendencias$ratioavg <- TRUE
PARAM$FE_hist$Tendencias$ratiomax <- FALSE


In [37]:

if( PARAM$FE_hist$Tendencias$run) {
  # 1. Ventana semestral (6 meses): tendencia y ratio sobre promedio 6m
  TendenciaYmuchomas(dataset,
    cols = cols_lagueables,
    ventana = 6,
    tendencia = TRUE,
    minimo = FALSE,
    maximo = FALSE,
    promedio = FALSE,
    ratioavg = TRUE,
    ratiomax = FALSE
  )

  # 2. Ventana trimestral (3 meses): tendencia reciente y caida brusca vs promedio 3m
  TendenciaYmuchomas(dataset,
    cols = cols_lagueables,
    ventana = 3,
    tendencia = TRUE,
    minimo = FALSE,
    maximo = FALSE,
    promedio = FALSE,
    ratioavg = TRUE,
    ratiomax = FALSE
  )

  # 3. Valvula de seguridad post-Tendencias para limpiar Inf y NaN de ratioavg3 y ratioavg6
  for (col in names(dataset)) {
    set(dataset, i = which(is.infinite(dataset[[col]])), j = col, value = NA)
    set(dataset, i = which(is.nan(dataset[[col]])), j = col, value = 0)
  }
}

Verificacion de los campos recien creados

In [38]:
ncol(dataset)
colnames(dataset)

[1] 1029

[1] "numero_de_cliente"                           
   [2] "foto_mes"                                    
   [3] "internet"                                    
   [4] "cliente_edad"                                
   [5] "cliente_antiguedad"                          
   [6] "mrentabilidad"                               
   [7] "mrentabilidad_annual"                        
   [8] "mcomisiones"                                 
   [9] "mactivos_margen"                             
  [10] "mpasivos_margen"                             
  [11] "cproductos"                                  
  [12] "mcuenta_corriente"                           
  [13] "mcaja_ahorro"                                
  [14] "cdescubierto_preacordado"                    
  [15] "mcuentas_saldo"                              
  [16] "ctarjeta_visa"                               
  [17] "ctarjeta_visa_transacciones"                 
  [18] "mtarjeta_visa_consumo"                       
  [19] "ctarjeta_master"                             
  [20] "ctarjeta_master_transacciones"               
  [21] "mtarjeta_master_consumo"                     
  [22] "cprestamos_personales"                       
  [23] "mprestamos_personales"                       
  [24] "cpayroll_trx"                                
  [25] "mpayroll"                                    
  [26] "mttarjeta_visa_debitos_automaticos"          
  [27] "ccomisiones_mantenimiento"                   
  [28] "mcomisiones_mantenimiento"                   
  [29] "ccomisiones_otras"                           
  [30] "mtransferencias_recibidas"                   
  [31] "ccallcenter_transacciones"                   
  [32] "thomebanking"                                
  [33] "chomebanking_transacciones"                  
  [34] "ctrx_quarter"                                
  [35] "Master_status"                               
  [36] "Master_mfinanciacion_limite"                 
  [37] "Master_Fvencimiento"                         
  [38] "Master_msaldototal"                          
  [39] "Master_mlimitecompra"                        
  [40] "Master_fultimo_cierre"                       
  [41] "Master_fechaalta"                            
  [42] "Master_mconsumototal"                        
  [43] "Master_cconsumos"                            
  [44] "Master_mpagominimo"                          
  [45] "Visa_status"                                 
  [46] "Visa_mfinanciacion_limite"                   
  [47] "Visa_Fvencimiento"                           
  [48] "Visa_msaldototal"                            
  [49] "Visa_mlimitecompra"                          
  [50] "Visa_fultimo_cierre"                         
  [51] "Visa_fechaalta"                              
  [52] "Visa_mconsumototal"                          
  [53] "Visa_cconsumos"                              
  [54] "Visa_mpagominimo"                            
  [55] "clase_ternaria"                              
  [56] "kmes"                                        
  [57] "ctrx_quarter_normalizado"                    
  [58] "mpayroll_sobre_edad"                         
  [59] "vm_status01"                                 
  [60] "vm_status02"                                 
  [61] "vm_status03"                                 
  [62] "vm_status04"                                 
  [63] "vm_status05"                                 
  [64] "vm_status06"                                 
  [65] "mv_status07"                                 
  [66] "vm_mfinanciacion_limite"                     
  [67] "vm_Fvencimiento"                             
  [68] "vm_msaldototal"                              
  [69] "vm_mlimitecompra"                            
  [70] "vm_fultimo_cierre"                           
  [71] "vm_fechaalta"                                
  [72] "vm_mconsumototal"                            
  [73] "vm_cconsumos"                                
  [74] "vm_mpagominimo"                              
  [75] 

#### 9.7.1.5  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest



In [39]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

Loading required package: lightgbm



In [40]:
AgregaVarRandomForest <- function() {

  cat( "inicio AgregaVarRandomForest()\n")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% PARAM$FE_rf$train$clase01_valor1,
      clase01 := 1L ]

  campos_buenos <- setdiff(
    colnames(dataset),
    c( "clase_ternaria", "clase01")
  )

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$FE_rf$train$training )]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    free_raw_data = FALSE
  )

  modelo <- lgb.train(
     data = dtrain,
     param = PARAM$FE_rf$lgb_param,
     verbose = -100
  )

  cat( "Fin construccion RandomForest\n" )
  # grabo el modelo, achivo .model
  lgb.save(modelo, file="modelo.model" )

  qarbolitos <- copy(PARAM$FE_rf$lgb_param$num_iterations)

  periodos <- dataset[ , unique( foto_mes ) ]

  for( periodo in  periodos )
  {
    cat( "periodo = ", periodo, "\n" )
    datamatrix <- data.matrix(dataset[ foto_mes== periodo, campos_buenos, with = FALSE])

    cat( "Inicio prediccion\n" )
    prediccion <- predict(
        modelo,
        datamatrix,
        type = "leaf"
    )
    cat( "Fin prediccion\n" )

    for( arbolito in 1:qarbolitos )
    {
       cat( arbolito, " " )
       hojas_arbol <- unique(prediccion[ , arbolito])

       for (pos in 1:length(hojas_arbol)) {
         # el numero de nodo de la hoja, estan salteados
         nodo_id <- hojas_arbol[pos]
         dataset[ foto_mes== periodo, paste0(
            "rf_", sprintf("%03d", arbolito),
             "_", sprintf("%03d", nodo_id)
          ) :=  as.integer( nodo_id == prediccion[ , arbolito]) ]

       }

       rm( hojas_arbol )
    }
    cat( "\n" )

    rm( prediccion )
    rm( datamatrix )
    gc(verbose= FALSE)
  }

  gc(verbose= FALSE)

  # borro clase01 , no debe ensuciar el dataset
  dataset[ , clase01 := NULL ]

}


In [41]:
# Parametros de Feature Engineering  a partir de hojas de Random Forest

# Estos CUATRO parametros son los que se deben modificar
PARAM$FE_rf$arbolitos= 20
PARAM$FE_rf$hojas_por_arbol= 16
PARAM$FE_rf$datos_por_hoja= 100
PARAM$FE_rf$mtry_ratio= 0.2

# Estos son quasi fijos
PARAM$FE_rf$train$clase01_valor1 <- c( "BAJA+2", "BAJA+1")
PARAM$FE_rf$train$training <- c( 202101, 202102, 202103)

# Estos TAMBIEN son quasi fijos
PARAM$FE_rf$lgb_param <-list(
    # parametros que se pueden cambiar
    num_iterations = PARAM$FE_rf$arbolitos,
    num_leaves  = PARAM$FE_rf$hojas_por_arbol,
    min_data_in_leaf = PARAM$FE_rf$datos_por_hoja,
    feature_fraction_bynode  = PARAM$FE_rf$mtry_ratio,

    # para que LightGBM emule Random Forest
    boosting = "rf",
    bagging_fraction = ( 1.0 - 1.0/exp(1.0) ),
    bagging_freq = 1L,
    feature_fraction = 1.0,

    # genericos de LightGBM
    max_bin = 31L,
    objective = "binary",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    force_row_wise = TRUE,
    verbosity = -100,
    max_depth = -1L,
    min_gain_to_split = 0.0,
    min_sum_hessian_in_leaf = 0.001,
    lambda_l1 = 0.0,
    lambda_l2 = 0.0,

    pos_bagging_fraction = 1.0,
    neg_bagging_fraction = 1.0,
    is_unbalance = FALSE,
    scale_pos_weight = 1.0,

    drop_rate = 0.1,
    max_drop = 50,
    skip_drop = 0.5,

    extra_trees = FALSE
  )

In [42]:
# Feature Engineering agregando variables de Random Forest
# AgregaVarRandomForest()

In [43]:
ncol(dataset)
colnames(dataset)

[1] 1029

[1] "numero_de_cliente"                           
   [2] "foto_mes"                                    
   [3] "internet"                                    
   [4] "cliente_edad"                                
   [5] "cliente_antiguedad"                          
   [6] "mrentabilidad"                               
   [7] "mrentabilidad_annual"                        
   [8] "mcomisiones"                                 
   [9] "mactivos_margen"                             
  [10] "mpasivos_margen"                             
  [11] "cproductos"                                  
  [12] "mcuenta_corriente"                           
  [13] "mcaja_ahorro"                                
  [14] "cdescubierto_preacordado"                    
  [15] "mcuentas_saldo"                              
  [16] "ctarjeta_visa"                               
  [17] "ctarjeta_visa_transacciones"                 
  [18] "mtarjeta_visa_consumo"                       
  [19] "ctarjeta_master"                             
  [20] "ctarjeta_master_transacciones"               
  [21] "mtarjeta_master_consumo"                     
  [22] "cprestamos_personales"                       
  [23] "mprestamos_personales"                       
  [24] "cpayroll_trx"                                
  [25] "mpayroll"                                    
  [26] "mttarjeta_visa_debitos_automaticos"          
  [27] "ccomisiones_mantenimiento"                   
  [28] "mcomisiones_mantenimiento"                   
  [29] "ccomisiones_otras"                           
  [30] "mtransferencias_recibidas"                   
  [31] "ccallcenter_transacciones"                   
  [32] "thomebanking"                                
  [33] "chomebanking_transacciones"                  
  [34] "ctrx_quarter"                                
  [35] "Master_status"                               
  [36] "Master_mfinanciacion_limite"                 
  [37] "Master_Fvencimiento"                         
  [38] "Master_msaldototal"                          
  [39] "Master_mlimitecompra"                        
  [40] "Master_fultimo_cierre"                       
  [41] "Master_fechaalta"                            
  [42] "Master_mconsumototal"                        
  [43] "Master_cconsumos"                            
  [44] "Master_mpagominimo"                          
  [45] "Visa_status"                                 
  [46] "Visa_mfinanciacion_limite"                   
  [47] "Visa_Fvencimiento"                           
  [48] "Visa_msaldototal"                            
  [49] "Visa_mlimitecompra"                          
  [50] "Visa_fultimo_cierre"                         
  [51] "Visa_fechaalta"                              
  [52] "Visa_mconsumototal"                          
  [53] "Visa_cconsumos"                              
  [54] "Visa_mpagominimo"                            
  [55] "clase_ternaria"                              
  [56] "kmes"                                        
  [57] "ctrx_quarter_normalizado"                    
  [58] "mpayroll_sobre_edad"                         
  [59] "vm_status01"                                 
  [60] "vm_status02"                                 
  [61] "vm_status03"                                 
  [62] "vm_status04"                                 
  [63] "vm_status05"                                 
  [64] "vm_status06"                                 
  [65] "mv_status07"                                 
  [66] "vm_mfinanciacion_limite"                     
  [67] "vm_Fvencimiento"                             
  [68] "vm_msaldototal"                              
  [69] "vm_mlimitecompra"                            
  [70] "vm_fultimo_cierre"                           
  [71] "vm_fechaalta"                                
  [72] "vm_mconsumototal"                            
  [73] "vm_cconsumos"                                
  [74] "vm_mpagominimo"                              
  [75] 

#### 9.7.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr*

El objetivo de esta etapa NO es mejorar el modelo predictivo

El objetivo es eliminar campos poco importantes para hacer espacio a nuevos campos, debido a las restricciones de memoria RAM.

In [44]:
VPOS_CORTE <- c()

fganancia_lgbm_meseta <- function(probs, datos) {
  vlabels <- get_field(datos, "label")
  vpesos <- get_field(datos, "weight")

  tbl <- as.data.table(list(
    "prob" = probs,
    "gan" = ifelse(vlabels == 1 & vpesos > 1, PARAM$CN$train$gan1, PARAM$CN$train$gan0)
  ))

  setorder(tbl, -prob)
  tbl[, posicion := .I]
  tbl[, gan_acum := cumsum(gan)]
  setorder(tbl, -gan_acum) # voy por la meseta

  gan <- mean(tbl[1:500, gan_acum]) # meseta de tamaño 500

  pos_meseta <- tbl[1:500, median(posicion)]
  VPOS_CORTE <<- c(VPOS_CORTE, pos_meseta)

  return(list(
    "name" = "ganancia",
    "value" = gan,
    "higher_better" = TRUE
  ))
}


In [45]:
# Elimina del dataset las variables que estan por debajo
#  de la capa geologica de canaritos
# se llama varias veces, luego de agregar muchas variables nuevas,
#  para ir reduciendo la cantidad de variables
# y así hacer lugar a nuevas variables importantes
# modifficación

GVEZ <- 1

campitos <- c( "numero_de_cliente", "foto_mes", "clase_ternaria" )

CanaritosAsesinos <- function(
  canaritos_ratio,
  canaritos_desvios,
  canaritos_semilla) {

  cat( "inicio CanaritosAsesinos()\n")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% PARAM$CN$train$clase01_valor1,
      clase01 := 1L ]

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  for (i in 1:(ncol(dataset) * canaritos_ratio)) {
    dataset[, paste0("canarito", i) := runif(nrow(dataset))]
  }

  campos_buenos <- setdiff(
    colnames(dataset),
    c( campitos, "clase01")
  )

  azar <- runif(nrow(dataset))

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$CN$train$training &
      (clase01 == 1 | azar < PARAM$CN$train$undersampling))]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    weight = dataset[
      entrenamiento == TRUE,
      ifelse(clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )

  dvalid <- lgb.Dataset(
    data = data.matrix(dataset[foto_mes %in% PARAM$CN$train$validation, campos_buenos, with = FALSE]),
    label = dataset[foto_mes %in% PARAM$CN$train$validation, clase01],
    weight = dataset[
      foto_mes %in% PARAM$CN$train$validation,
      ifelse( clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )


  param <- list(
    objective = "binary",
    metric = "custom",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    verbosity = -100,
    seed = canaritos_semilla,
    max_depth = -1, # -1 significa no limitar,  por ahora lo dejo fijo
    min_gain_to_split = 0.0, # por ahora, lo dejo fijo
    lambda_l1 = 0.0, # por ahora, lo dejo fijo
    lambda_l2 = 0.0, # por ahora, lo dejo fijo
    max_bin = 31, # por ahora, lo dejo fijo
    num_iterations = 9999, # un numero grande, lo limita early_stopping_rounds
    force_row_wise = TRUE, # para que los alumnos no se atemoricen con  warning
    learning_rate = 0.065,
    feature_fraction = 0.5, 
    min_data_in_leaf = 260,
    num_leaves = 60,
    early_stopping_rounds = 200,
    num_threads = parallel::detectCores()
  )

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  modelo <- lgb.train(
    data = dtrain,
    valids = list(valid = dvalid),
    eval = fganancia_lgbm_meseta,
    param = param,
    verbose = -100
  )

  tb_importancia <- lgb.importance(model = modelo)
  tb_importancia[, pos := .I]

  fwrite(tb_importancia,
    file = paste0("impo_", GVEZ, ".txt"),
    sep = "\t"
  )

  GVEZ <<- GVEZ + 1

  umbral <- tb_importancia[
    Feature %like% "canarito",
    median(pos) + canaritos_desvios * sd(pos)
  ] # Atencion corto en la mediana mas desvios!!

  col_utiles <- tb_importancia[
    pos < umbral & !(Feature %like% "canarito"),
    Feature
  ]

  col_utiles <- unique(c(
    col_utiles,
    c(campitos, cols_lagueables, "kmes")
  ))

  col_inutiles <- setdiff(colnames(dataset), col_utiles)

  dataset[, (col_inutiles) := NULL]

  cat( "fin CanaritosAsesinos()\n")

  return( tb_importancia )
}


In [46]:
# Estos DOS parametros son los que se deben modificar (modificación)
# En Celda 47 (Parametros de Canaritos alineados a 202107):
PARAM$CN$ratio <- 0.2
PARAM$CN$desvios <- 1

PARAM$CN$train$clase01_valor1 <- c( "BAJA+2", "BAJA+1")
PARAM$CN$train$positivos <- c( "BAJA+2")
PARAM$CN$train$training <- c( 202103, 202104, 202105)
PARAM$CN$train$validation <- c( 202107 )
PARAM$CN$train$undersampling <- 1.0
PARAM$CN$train$gan1 <- 0.975
PARAM$CN$train$gan0 <- -0.025

In [47]:
# la llamada a Canaritos Asesinos
tb_importancia <- CanaritosAsesinos(
  canaritos_ratio = PARAM$CN$ratio,
  canaritos_desvios = PARAM$CN$desvios,
  canaritos_semilla = PARAM$semilla_primigenia
)


inicio CanaritosAsesinos()
fin CanaritosAsesinos()


In [48]:
# grabo la importancia, ver el archivo directamente en la carpeta

fwrite( tb_importancia,
  file="canaritos.txt",
  sep="\t"
)

In [49]:
# verifico
ncol(dataset)
colnames(dataset)

[1] 304

[1] "numero_de_cliente"                      
  [2] "foto_mes"                               
  [3] "internet"                               
  [4] "cliente_edad"                           
  [5] "cliente_antiguedad"                     
  [6] "mrentabilidad"                          
  [7] "mrentabilidad_annual"                   
  [8] "mcomisiones"                            
  [9] "mactivos_margen"                        
 [10] "mpasivos_margen"                        
 [11] "cproductos"                             
 [12] "mcuenta_corriente"                      
 [13] "mcaja_ahorro"                           
 [14] "cdescubierto_preacordado"               
 [15] "mcuentas_saldo"                         
 [16] "ctarjeta_visa"                          
 [17] "ctarjeta_visa_transacciones"            
 [18] "mtarjeta_visa_consumo"                  
 [19] "ctarjeta_master"                        
 [20] "ctarjeta_master_transacciones"          
 [21] "mtarjeta_master_consumo"                
 [22] "cprestamos_personales"                  
 [23] "mprestamos_personales"                  
 [24] "cpayroll_trx"                           
 [25] "mpayroll"                               
 [26] "mttarjeta_visa_debitos_automaticos"     
 [27] "ccomisiones_mantenimiento"              
 [28] "mcomisiones_mantenimiento"              
 [29] "ccomisiones_otras"                      
 [30] "mtransferencias_recibidas"              
 [31] "ccallcenter_transacciones"              
 [32] "thomebanking"                           
 [33] "chomebanking_transacciones"             
 [34] "ctrx_quarter"                           
 [35] "Master_status"                          
 [36] "Master_mfinanciacion_limite"            
 [37] "Master_Fvencimiento"                    
 [38] "Master_msaldototal"                     
 [39] "Master_mlimitecompra"                   
 [40] "Master_fultimo_cierre"                  
 [41] "Master_fechaalta"                       
 [42] "Master_mconsumototal"                   
 [43] "Master_cconsumos"                       
 [44] "Master_mpagominimo"                     
 [45] "Visa_status"                            
 [46] "Visa_mfinanciacion_limite"              
 [47] "Visa_Fvencimiento"                      
 [48] "Visa_msaldototal"                       
 [49] "Visa_mlimitecompra"                     
 [50] "Visa_fultimo_cierre"                    
 [51] "Visa_fechaalta"                         
 [52] "Visa_mconsumototal"                     
 [53] "Visa_cconsumos"                         
 [54] "Visa_mpagominimo"                       
 [55] "clase_ternaria"                         
 [56] "kmes"                                   
 [57] "ctrx_quarter_normalizado"               
 [58] "mpayroll_sobre_edad"                    
 [59] "vm_status01"                            
 [60] "vm_status02"                            
 [61] "vm_status03"                            
 [62] "vm_status04"                            
 [63] "vm_status05"                            
 [64] "vm_status06"                            
 [65] "mv_status07"                            
 [66] "vm_mfinanciacion_limite"                
 [67] "vm_Fvencimiento"                        
 [68] "vm_msaldototal"                         
 [69] "vm_mlimitecompra"                       
 [70] "vm_fultimo_cierre"                      
 [71] "vm_fechaalta"                           
 [72] "vm_mconsumototal"                       
 [73] "vm_cconsumos"                           
 [74] "vm_mpagominimo"                         
 [75] "vmr_Master_mlimitecompra"               
 [76] "vmr_Visa_mlimitecompra"                 
 [77] "vmr_msaldototal"                        
 [78] "vmr_mconsumototal"                      
 [79] "vmr_mpagominimo"                        
 [80] "vm_ctarjetas_activas"                   
 [81] "vm_ctrx_banco"                          
 [82] "vm_mconsumo_banco"                      
 [83] "vmr_ticket_medio_banco"                 
 [84] "vmr_consumo

In [50]:
# Checkpoint: guardo el dataset limpio post-Canaritos (modificación)
fwrite(dataset, file = "dataset_post_canaritos.csv.gz")

### 9.7.2 Modelado

#### 9.7.2.1 Training Strategy


Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ] sin undersampling de los CONTINUA

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 20% de los CONTINUA

In [51]:
# modificación del pct
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)

PARAM$trainingstrategy$training_pct <- 0.5

PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [52]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [53]:
# los campos en los que se entrena (modificación)
# los campos en los que se entrena (excluimos numero_de_cliente y foto_mes)
campos_buenos <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
) )

In [54]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [55]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32938

####  9.7.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Cantidad de iteraciones inteligentes de la Optimizacion Bayesiana = **40**

In [56]:
# paquetes necesarios para la Bayesian Optimization
if(!require("DiceKriging")) install.packages("DiceKriging")
require("DiceKriging")

if(!require("mlrMBO")) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: DiceKriging

Loading required package: mlrMBO

Loading required package: mlr

Loading required package: ParamHelpers


Attaching package: ‘ParamHelpers’


The following object is masked from ‘package:R.utils’:

    isVector



Attaching package: ‘mlr’


The following objects are masked from ‘package:R.utils’:

    resample, setThreshold


Loading required package: smoof

Loading required package: checkmate


Attaching package: ‘checkmate’


The following object is masked from ‘package:DiceKriging’:

    checkNames


The following object is masked from ‘package:R.utils’:

    asInt



Attaching package: ‘smoof’


The following objects are masked from ‘package:R.oo’:

    getDescription, getName




Definición de la Bayesian Optimization
<br> Si se desea optimizar un hiperparámetro que esta como fijo, debe QUITARSE de param_fijos y agregarse a PARAM$hipeparametertuning$hs

In [57]:
# 50 iteraciones inteligentes (modificación) 
# CELDA DE param_fijos y hs (9.7.2.2) - CORREGIDA Y SINCRONIZADA:
PARAM$hipeparametertuning$num_interations <- 50

PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE,
  seed= PARAM$semilla_primigenia,
  extra_trees= FALSE,

  num_iterations= 2048,        # CORREGIDO de 200 a 2048! Corta con early_stopping_rounds
  early_stopping_rounds= 200,

  max_depth= -1L,
  min_gain_to_split= 0.0,
  bagging_freq= 1L,            # Necesario para que funcione bagging_fraction

  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  max_bin= 31,
  num_threads= parallel::detectCores()
)

PARAM$hipeparametertuning$hs <- makeParamSet(
  makeNumericParam("num_leaves", lower= 4.0, upper= 8.5, trafo= function(x) round(2^x) ),
  makeNumericParam("learning_rate", lower= -6.5, upper= -3.0, trafo= function(x) 2^x ),
  makeNumericParam("feature_fraction", lower= 0.25, upper= 0.80),
  makeNumericParam("min_data_in_leaf", lower= 6.0, upper= 10.5, trafo= function(x) round(2^x) ),
  makeNumericParam("min_sum_hessian_in_leaf", lower= -8.0, upper= 5.0, trafo= function(x) 2^x),
  makeNumericParam("bagging_fraction", lower= 0.60, upper= 1.0),
  makeNumericParam("lambda_l1", lower= 0.0, upper= 30.0),
  makeNumericParam("lambda_l2", lower= 0.0, upper= 60.0)
)



Función "señora caja negra"  que es llamada para verificar la realidad por la Bayesian Optimization

In [58]:
# En  x llegan los parmaetros de la bayesiana
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando
# modificacion

# CELDA DE EstimarGanancia_AUC_lightgbm (9.7.2.2):
# Guardamos el best_iter exacto en un vector global para recuperarlo al final
V_BEST_ITER <- c()

EstimarGanancia_AUC_lightgbm <- function(x) {

  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]
  V_BEST_ITER <<- c(V_BEST_ITER, modelo_train$best_iter)

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " best_iter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return(AUC)
}

seteo de la Bayesian Optimization (complejo)
<br> copiado y pegado de la documentación de la librería

In [59]:
configureMlr(show.learner.output = FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo
obj.fun <- makeSingleObjectiveFunction(
    fn= EstimarGanancia_AUC_lightgbm, # la funcion que voy a maximizar
    minimize= FALSE, # estoy Maximizando AUC
    noisy= FALSE,
    par.set= PARAM$hipeparametertuning$hs,
    has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
    save.on.disk.at.time= 600,
    save.file.path= "HT.RDATA"
)

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
    ctrl,
    iters= PARAM$hipeparametertuning$num_interations  # cantidad de iteraciones inteligentes
)

# defino el método estandar para la creacion de los puntos iniciales
#   los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit = makeMBOInfillCritEI())

# mas configuraciones
surr.km <- makeLearner(
    "regr.km",
    predict.type= "se",
    covtype= "matern3_2",
    control= list(trace = TRUE)
)

Corrida de la Bayesian Optimization,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer

Si corrío a medias y llegó a las iteraciones inteligentes, en el archivo binario HT.RDATA quedó lo ya procesado y es utilizado para retomar la corrida desde lo último que llegó a grabar.

In [60]:
# inicio la optimizacion bayesiana

if (!file.exists("HT.RDATA")) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue("HT.RDATA") # retomo en caso que ya exista
}

Computing y column(s) for design. Not provided.

Wed Nov 11 21:39:58 2026  57, 0.0507893701618832, 0.421887942974981, 496, 8.45792525500726, 0.842014397101941, 14.0236133191587, 22.7491659436669 best_iter 412 AUC 0.934965165102295

Wed Nov 11 21:41:18 2026  103, 0.0165832551529817, 0.477414585249946, 827, 0.0924200145234712, 0.884114382138544, 26.7591672858472, 40.8350699400633 best_iter 1498 AUC 0.934518097863382

Wed Nov 11 21:41:37 2026  221, 0.0802164644234525, 0.406319050074014, 178, 0.321024317775538, 0.783121224711466, 20.3411024874395, 13.0276582078084 best_iter 198 AUC 0.934344227367361

Wed Nov 11 21:42:27 2026  31, 0.0226223912852712, 0.646130839825833, 133, 2.4397014555923, 0.757416471691715, 15.0738199135328, 36.6601831513266 best_iter 934 AUC 0.934863393698315

Wed Nov 11 21:43:32 2026  90, 0.0128137048649508, 0.593404630603991, 283, 0.0619913739887104, 0.73190023283771, 11.2155392728937, 32.7764326597385 best_iter 968 AUC 0.934571428722481

Wed Nov 11 21:44:55 2026  240,

la bayesian optimization ha corrido, extraigo los mejores hiperparametros

In [61]:
# almaceno los resultados de la Bayesian Optimization
# y capturo los mejores hiperparametros encontrados
# modificacion

# CELDA DE extraccion de tb_bayesiana (justo despues de mbo):
tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)
tb_bayesiana[, num_iterations := V_BEST_ITER]

setorder(tb_bayesiana, -y, -num_iterations)

fwrite( tb_bayesiana,
  file="BO_log.txt",
  sep="\t"
)

PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1,
  setdiff(colnames(tb_bayesiana),
    c("y","dob","eol","error.message","exec.time","ei","error.model",
      "train.time","prop.type","propose.time","se","mean","iter")),
  with= FALSE
]

print(PARAM$out$lgbm$mejores_hiperparametros)

   num_leaves learning_rate feature_fraction min_data_in_leaf
        <num>         <num>            <num>            <num>
1:   6.274205     -5.331875        0.2606967         6.634543
   min_sum_hessian_in_leaf bagging_fraction lambda_l1 lambda_l2 num_iterations
                     <num>            <num>     <num>     <num>          <int>
1:                -6.20921        0.9252376  1.941347  4.517969            273


### 9.7.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización bayesiana

In [62]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 910853

##### Final Training Hyperparameters

In [63]:
# Limpio la memoria de los datasets de la BO (modificación)
rm(dtrain, dvalidate)
gc(full = TRUE, verbose = FALSE)

fijos <- copy(PARAM$lgbm$param_fijos)
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

param_final <- copy(fijos)

x <- PARAM$out$lgbm$mejores_hiperparametros
param_final$num_iterations <- x$num_iterations
param_final$num_leaves <- round( 2^x$num_leaves )
param_final$learning_rate <- 2^x$learning_rate
param_final$feature_fraction <- x$feature_fraction
param_final$min_data_in_leaf <- round( 2^x$min_data_in_leaf )
param_final$min_sum_hessian_in_leaf <- 2^x$min_sum_hessian_in_leaf
param_final$bagging_fraction <- x$bagging_fraction
param_final$lambda_l1 <- x$lambda_l1
param_final$lambda_l2 <- x$lambda_l2



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,3057419,163.3,5018178,268,5018178,268.0
Vcells,571376630,4359.3,907016432,6920,1667952545,12725.5


In [64]:
param_final

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] TRUE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$verbosity
[1] -100

$force_row_wise
[1] TRUE

$seed
[1] 415517

$extra_trees
[1] FALSE

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$bagging_freq
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$max_bin
[1] 31

$num_threads
[1] 16

$num_iterations
[1] 273

$num_leaves
[1] 77

$learning_rate
[1] 0.02482822

$feature_fraction
[1] 0.2606967

$min_data_in_leaf
[1] 99

$min_sum_hessian_in_leaf
[1] 0.0135158

$bagging_fraction
[1] 0.9252376

$lambda_l1
[1] 1.941347

$lambda_l2
[1] 4.517969

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [65]:
# poner este valor en 5 o en 1  si se está muy ajustado de tiempo (modificación)

PARAM$FT$semillerio <- 50  # cantidad de semillas

In [66]:
if(!require("primes")) install.packages("primes")
require("primes")

Loading required package: primes



In [67]:
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
# me quedo con PARAM$semillerio  primos al azar
PARAM$FT$semillas <- sample(primos)[seq(PARAM$FT$semillerio)]

cat( PARAM$FT$semillas)

145423 650543 142151 857723 781799 129803 485701 457381 570181 132619 649739 387437 254027 671383 974249 661099 834341 102769 484609 856241 904573 511933 622177 330287 209813 240011 321073 129793 387683 229253 392519 284803 278689 760897 671999 632923 640873 979439 254899 261241 555697 937927 503267 271597 363329 212209 656459 660281 691813 457673

In [68]:
primero <- TRUE

crear_modelo_final <- function( sem ) {

  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  if( !file.exists(nombre_arch) )
  {
    param_final$seed <- sem

    set.seed(sem, kind = "L'Ecuyer-CMRG")
    final_model <- lgb.train(
      data= dfinal_train,
      param= param_final,
      verbose= -100
    )

    lgb.save(final_model, nombre_arch) # grabo el modelo"

    # grabo la primer importancia de variables
    #  Natalia : da lo mismo cual se guarda
    if( primero)
    {
      primero <<- FALSE
      tb_importancia <- as.data.table(lgb.importance(final_model))
      archivo_importancia <- "impo.txt"

      fwrite( tb_importancia,
        file= archivo_importancia,
        sep= "\t"
      )
    }

    rm(final_model)
    gc(full = TRUE, verbose=FALSE)
  }
}

In [69]:
gc(full = TRUE, verbose=FALSE)
dir.create("modelos", showWarnings =FALSE)

primero <- TRUE
for( sem in PARAM$FT$semillas)  crear_modelo_final(sem)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,3059407,163.4,5018178,268,5018178,268.0
Vcells,570954579,4356.1,907016432,6920,1667952545,12725.5


#### Scoring

Aplico el modelo final a los datos del futuro

In [70]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [71]:
# aplico final_model   a dfuture

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := 0]

datos_matrix <- data.matrix(dfuture[, campos_buenos, with= FALSE])

In [72]:
agregar_a_ensemble <- function( sem ) {

  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  final_model <- lgb.load(nombre_arch)

  prediccion <- predict(
    final_model,
    datos_matrix
  )

  tb_prediccion[, paste0("prob_", isem) := prediccion]
  tb_prediccion[, prob := prob + prediccion]

  rm(final_model)
  rm(prediccion)
  gc(full = TRUE, verbose=FALSE)
}

In [73]:
for( isem in seq(length(PARAM$FT$semillas)) )
{
  sem <- PARAM$FT$semillas[ isem ]
  agregar_a_ensemble( sem )
  gc(full = TRUE, verbose=FALSE)
}

rm( datos_matrix)
gc(full = TRUE, verbose=FALSE)

tb_prediccion[, prob := prob /length(PARAM$FT$semillas) ]

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,3072457,164.1,5018178,268,5018178,268.0
Vcells,307948806,2349.5,907016432,6920,1667952545,12725.5


In [74]:
# veo que hay en tb_prediccion
tb_prediccion

numero_de_cliente,prob,prob_1,prob_2,prob_3,prob_4,prob_5,prob_6,prob_7,prob_8,⋯,prob_41,prob_42,prob_43,prob_44,prob_45,prob_46,prob_47,prob_48,prob_49,prob_50
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
11675570,0.0010732399,0.0014408560,0.0009138397,0.0009530236,0.0010664499,0.0010477111,0.0013290858,0.0011844516,0.0013404284,⋯,0.0013154952,0.0010849512,0.0010047541,0.0009130273,0.0009328161,0.0011518922,0.0011521158,0.0010436518,0.0010740284,0.0009898432
11675731,0.0009388741,0.0011092072,0.0009741171,0.0009677928,0.0008242356,0.0008527472,0.0008166483,0.0010090374,0.0008004797,⋯,0.0008516958,0.0009789165,0.0011084525,0.0008608572,0.0009477016,0.0008951449,0.0008906234,0.0009312705,0.0010072232,0.0010285854
11677630,0.0037252796,0.0037328405,0.0041578761,0.0039406777,0.0028524951,0.0044887967,0.0036046027,0.0037808601,0.0033180399,⋯,0.0033966641,0.0036894426,0.0033608351,0.0037951384,0.0029661564,0.0040100171,0.0038263389,0.0039859392,0.0038814524,0.0036737309
11678491,0.0008789614,0.0007830754,0.0007299495,0.0007926806,0.0009191744,0.0008146900,0.0009101697,0.0009175025,0.0008657109,⋯,0.0008159750,0.0008165548,0.0008298754,0.0009192861,0.0008823304,0.0009269501,0.0007606423,0.0011203398,0.0009343980,0.0008008211
11678621,0.0008947336,0.0009025803,0.0008386733,0.0009476421,0.0009639985,0.0007545144,0.0009713277,0.0007446161,0.0009111449,⋯,0.0010773168,0.0009678883,0.0009321747,0.0008554610,0.0007624009,0.0010359346,0.0008097088,0.0009797174,0.0010251548,0.0007421856
11679331,0.0009468945,0.0007137718,0.0006508031,0.0012370928,0.0013033073,0.0010871582,0.0009858670,0.0010849227,0.0008372144,⋯,0.0009618284,0.0009540770,0.0010036456,0.0009081633,0.0006848451,0.0008972440,0.0007667662,0.0009478012,0.0008958547,0.0009378502
11680281,0.0015039610,0.0012853415,0.0014738762,0.0014724656,0.0019136589,0.0017480762,0.0013330530,0.0015841049,0.0013667866,⋯,0.0016858615,0.0016877393,0.0013039786,0.0015380885,0.0016682056,0.0016510811,0.0012423495,0.0017883770,0.0016834337,0.0015885298
11682860,0.0002698685,0.0002734564,0.0002770615,0.0002218574,0.0002693680,0.0002573345,0.0002425579,0.0002516620,0.0002884088,⋯,0.0002424361,0.0002622003,0.0002899027,0.0003149720,0.0002807833,0.0002482644,0.0003071768,0.0002438885,0.0002609053,0.0002660353
11683331,0.0119863661,0.0101186763,0.0117166925,0.0105261033,0.0122132127,0.0091500107,0.0126848427,0.0124987723,0.0122758156,⋯,0.0093675173,0.0114823726,0.0115481013,0.0106315614,0.0116856125,0.0149106429,0.0117835903,0.0104595884,0.0150557604,0.0128572683


In [75]:
# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggles

In [76]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle (modificación a junior)

PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-jr"
PARAM$kaggle$cortes <- seq(1600, 2300, by = 100)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

  archivo_kaggle <- paste0(
    "./kaggle/KA",
    PARAM$experimento, "_",
    "s", length(PARAM$FT$semillas), "_",
    envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

In [77]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

Loading required package: yaml



In [78]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Wed Nov 11 22:37:45 2026"